In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# StockLens/ repo root (filter/ -> feature_selection/ -> notebooks/ -> root)
PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.mrmr import mrmr_greedy

FILTER_RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "filter_results"

# These now come from the leakage-fixed mutual_information.ipynb run
# (train split only).
feature_target_mi = pd.read_csv(FILTER_RESULTS_DIR / "feature_target_mi.csv")
feature_mi_matrix = pd.read_csv(FILTER_RESULTS_DIR / "feature_mi_matrix.csv")

print("Feature-Target MI")
display(feature_target_mi.head())

print("Feature MI Matrix")
display(feature_mi_matrix.head())


Feature-Target MI


,feature,mi_score
0,rsi_14,0.070074
1,volatility_20,0.064134
2,macd_signal,0.056946
3,volume_sma_20,0.055255
4,atr_14,0.050383


Feature MI Matrix


,Unnamed: 0,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,roc_20,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20
0,return_1d,0.000000,0.123459,0.065863,0.053170,0.646476,0.184390,0.205498,0.134822,0.054703,...,0.053170,0.000000,0.000510,0.029407,0.133122,0.050404,0.024989,0.122212,0.061600,0.150318
1,return_5d,0.123459,0.000000,0.362350,0.133865,0.040889,0.036107,0.065457,0.019686,0.020814,...,0.133865,0.079196,0.057141,0.255325,0.140954,0.098751,0.042795,0.000000,0.072301,0.095413
2,return_10d,0.065863,0.362350,0.000000,0.390133,0.036547,0.090082,0.048655,0.128769,0.110519,...,0.390133,0.212396,0.102393,0.582723,0.104611,0.172882,0.142541,0.003694,0.174118,0.057390
3,return_20d,0.053170,0.133865,0.390133,0.000000,0.001533,0.036861,0.061230,0.259545,0.185778,...,6.279094,0.651503,0.403877,0.247327,0.148561,0.325734,0.266493,0.000000,0.267052,0.072073
4,intraday_return,0.646476,0.040889,0.036547,0.001533,0.000000,0.541085,0.029510,0.127586,0.092206,...,0.001415,0.004067,0.000000,0.000000,0.067046,0.039757,0.033459,0.037704,0.016560,0.114446


In [2]:
selected_features = mrmr_greedy(
    feature_target_mi=feature_target_mi,
    feature_mi_matrix=feature_mi_matrix,
    k=10
)

print("Selected Features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")


Selected Features:
1. rsi_14
2. gap
3. high_low_range
4. volatility_20
5. volume_change_1d
6. volume_ratio_20
7. volatility_5
8. return_5d
9. return_1d
10. macd_signal


In [3]:
mrmr_results = pd.DataFrame({
    "rank": range(1, len(selected_features) + 1),
    "feature": selected_features
})

mrmr_results.to_csv(
    FILTER_RESULTS_DIR / "mrmr_results.csv",
    index=False
)


In [4]:
relevance = dict(
    zip(
        feature_target_mi["feature"],
        feature_target_mi["mi_score"]
    )
)

mi_matrix = feature_mi_matrix.set_index(
    feature_mi_matrix.columns[0]
)

validation_results = []

for rank, feature in enumerate(selected_features, start=1):

    if rank == 1:
        redundancy = 0.0
    else:
        previous_features = selected_features[:rank - 1]

        redundancy = np.mean([
            mi_matrix.loc[feature, prev]
            for prev in previous_features
        ])

    mrmr_score = relevance[feature] - redundancy

    validation_results.append({
        "rank": rank,
        "feature": feature,
        "relevance": relevance[feature],
        "redundancy": redundancy,
        "mrmr_score": mrmr_score
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)


,rank,feature,relevance,redundancy,mrmr_score
0,1,rsi_14,0.070074,0.000000,0.070074
1,2,gap,0.015860,0.038200,-0.022339
2,3,high_low_range,0.000000,0.029807,-0.029807
3,4,volatility_20,0.064134,0.111026,-0.046892
4,5,volume_change_1d,0.001885,0.039679,-0.037793
5,6,volume_ratio_20,0.019291,0.102016,-0.082724
6,7,volatility_5,0.029891,0.122823,-0.092932
7,8,return_5d,0.010298,0.114400,-0.104102
8,9,return_1d,0.025765,0.130661,-0.104896
9,10,macd_signal,0.056946,0.145848,-0.088903
